In [1]:
import pandas as pd

print("Operación: Rescate de variable de parque vehicular...")

# 1. Cargar tu base actual de la Fase 1 (la que no tiene autos)
ruta_base_actual = r"..\data\processed\datos_modelo_final.csv"
df_actual = pd.read_csv(ruta_base_actual)

# 2. Cargar la base cruda original de hogares de la ENIGH (donde vive num_auto)
# Usamos low_memory=False por el peso del archivo
ruta_hogares_raw = r"..\data\raw\conjunto_de_datos_hogares_enigh2022_ns\conjunto_de_datos\conjunto_de_datos_hogares_enigh2022_ns.csv"
df_hogares = pd.read_csv(ruta_hogares_raw, low_memory=False)

# 3. Extraer solo las llaves maestras y la variable de autos
# Convertimos las llaves a string para que el cruce sea exacto
df_autos = df_hogares[['folioviv', 'foliohog', 'num_auto']].copy()
df_autos['folioviv'] = df_autos['folioviv'].astype(str)
df_autos['foliohog'] = df_autos['foliohog'].astype(str)

df_actual['folioviv'] = df_actual['folioviv'].astype(str)
df_actual['foliohog'] = df_actual['foliohog'].astype(str)

# 4. Limpieza previa y Fusión (Merge)
if 'num_auto' in df_actual.columns:
    df_actual = df_actual.drop(columns=['num_auto'])

df_corregido = pd.merge(df_actual, df_autos, on=['folioviv', 'foliohog'], how='left')

# Rellenar vacíos con 0 (hogares sin auto)
df_corregido['num_auto'] = df_corregido['num_auto'].fillna(0)

# 5. Sobrescribir la base procesada para que quede actualizada permanentemente
df_corregido.to_csv(ruta_base_actual, index=False)

print(f"¡Variable 'num_auto' inyectada con éxito!")
print(f"Total de registros actualizados: {len(df_corregido)}")

Operación: Rescate de variable de parque vehicular...
¡Variable 'num_auto' inyectada con éxito!
Total de registros actualizados: 10052


In [3]:
import pandas as pd
import numpy as np
import glob
import os
import statsmodels.api as sm

print("1. Reconstruyendo Matriz de Capital (Demanda + Oferta + Vehículos)...")

# --- A. DEMANDA ---
ruta_enigh = r"..\data\processed\datos_modelo_final.csv"
enigh = pd.read_csv(ruta_enigh)

# Verificar que el rescate de autos funcionó
if 'num_auto' not in enigh.columns:
    print("ALERTA: La variable 'num_auto' no está. Vuelve a correr el script de rescate.")
else:
    enigh['ingreso_mercado'] = enigh['ing_cor'] * enigh['factor']
    enigh['autos_mercado'] = enigh['num_auto'] * enigh['factor']

    demanda_mun = enigh.groupby('ubica_geo').apply(
        lambda x: pd.Series({
            'poblacion_hogares': x['factor'].sum(),
            'ingreso_total_zona': x['ingreso_mercado'].sum(),
            'parque_vehicular': x['autos_mercado'].sum()
        })
    ).reset_index()
    demanda_mun['ingreso_promedio_hogar'] = demanda_mun['ingreso_total_zona'] / demanda_mun['poblacion_hogares']

    # --- B. OFERTA ---
    ruta_raw = r"..\data\raw"
    archivos_denue = glob.glob(os.path.join(ruta_raw, '**', 'denue_inegi_*.csv'), recursive=True)
    
    lista_df = []
    for archivo in archivos_denue:
        df_temp = pd.read_csv(archivo, encoding='latin1', low_memory=False)
        df_temp.columns = df_temp.columns.str.lower()
        lista_df.append(df_temp)
    denue = pd.concat(lista_df, ignore_index=True)
    
    denue['ubica_geo'] = (denue['cve_ent'].astype(str).str.zfill(2) + denue['cve_mun'].astype(str).str.zfill(3)).astype(int)
    denue['competencia_club'] = np.where(denue['nom_estab'].str.upper().fillna('').str.contains("SAM'S|SAMS|CITY CLUB"), 1, 0)
    denue['es_b2b'] = np.where(denue['codigo_act'].astype(str).str.startswith(('722', '462112', '461110')), 1, 0)
    
    oferta_mun = denue.groupby('ubica_geo').agg(
        total_competidores=('competencia_club', 'sum'),
        masa_negocios_b2b=('es_b2b', 'sum')
    ).reset_index()

    # --- C. FUSIÓN ---
    matriz_capital = pd.merge(demanda_mun, oferta_mun, on='ubica_geo', how='inner')
    
    print("Matriz lista. Entrenando Modelo Logit Completo...\n")
    
    # --- D. LOGIT MODEL ---
    matriz_capital['ingreso_miles'] = matriz_capital['ingreso_promedio_hogar'] / 1000
    matriz_capital['b2b_miles'] = matriz_capital['masa_negocios_b2b'] / 1000
    matriz_capital['autos_miles'] = matriz_capital['parque_vehicular'] / 1000
    matriz_capital['Y_mercado_probado'] = np.where(matriz_capital['total_competidores'] > 0, 1, 0)

    variables = ['ingreso_miles', 'b2b_miles', 'autos_miles']
    X = sm.add_constant(matriz_capital[variables])
    Y = matriz_capital['Y_mercado_probado']

    try:
        modelo = sm.Logit(Y, X).fit(disp=False)
        
        print("\n" + "="*80)
        print("REPORTE ECONOMÉTRICO FINAL (Con Movilidad Logística)")
        print("="*80)
        print(modelo.summary())
        
        matriz_capital['prob_teorica'] = modelo.predict(X)
        ops = matriz_capital[(matriz_capital['prob_teorica'] > 0.40) & (matriz_capital['total_competidores'] == 0)]
        
        print("\n" + "="*80)
        print("🎯 TOP MUNICIPIOS 'OCÉANO AZUL' (ACTUALIZADO)")
        print("="*80)
        display(ops[['ubica_geo', 'ingreso_promedio_hogar', 'masa_negocios_b2b', 'parque_vehicular', 'prob_teorica']].sort_values(by='prob_teorica', ascending=False))
    except Exception as e:
        print(f"Error en estimación: {e}")

1. Reconstruyendo Matriz de Capital (Demanda + Oferta + Vehículos)...


C:\Users\hugod\AppData\Local\Temp\ipykernel_12944\2809997040.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  enigh['ingreso_mercado'] = enigh['ing_cor'] * enigh['factor']
C:\Users\hugod\AppData\Local\Temp\ipykernel_12944\2809997040.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  enigh['autos_mercado'] = enigh['num_auto'] * enigh['factor']


Matriz lista. Entrenando Modelo Logit Completo...


REPORTE ECONOMÉTRICO FINAL (Con Movilidad Logística)
                           Logit Regression Results                           
Dep. Variable:      Y_mercado_probado   No. Observations:                   98
Model:                          Logit   Df Residuals:                       94
Method:                           MLE   Df Model:                            3
Date:                Tue, 25 Aug 2026   Pseudo R-squ.:                  0.5239
Time:                        18:33:40   Log-Likelihood:                -22.948
converged:                       True   LL-Null:                       -48.196
Covariance Type:            nonrobust   LLR p-value:                 6.261e-11
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -4.2923      1.288     -3.334      0.001      -6.816      -1.769
ingreso_miles    

,ubica_geo,ingreso_promedio_hogar,masa_negocios_b2b,parque_vehicular,prob_teorica
36,14101,68376.161543,4773,66853.0,0.853188
58,19021,73976.541204,4116,74135.0,0.823658
65,19031,70300.266333,4043,29763.0,0.583248


In [4]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score

print("Iniciando Pruebas de Rigor Econométrico...\n")

# 1. Cargar la matriz actualizada (Asegúrate de que la ruta sea correcta)
ruta = r"..\data\processed\matriz_espacial_fase2.csv"
try:
    df = pd.read_csv(ruta)
except FileNotFoundError:
    print("Error: No se encontró la matriz. Asegúrate de haber guardado el CSV en el paso anterior.")
    raise

# 2. Reconstruir Variables
df['ingreso_miles'] = df['ingreso_promedio_hogar'] / 1000
df['b2b_miles'] = df['masa_negocios_b2b'] / 1000

# Verificamos si los autos están en la base (por si acaso)
if 'parque_vehicular' in df.columns:
    df['autos_miles'] = df['parque_vehicular'] / 1000
    variables = ['ingreso_miles', 'b2b_miles', 'autos_miles']
else:
    variables = ['ingreso_miles', 'b2b_miles']
    print("Nota: Variable 'parque_vehicular' no encontrada, usando modelo base.")

df['Y_mercado_probado'] = np.where(df['total_competidores'] > 0, 1, 0)

X = df[variables]
X_con_constante = sm.add_constant(X)
Y = df['Y_mercado_probado']

# --- PRUEBA 1: MULTICOLINEALIDAD (VIF) ---
print("="*60)
print("1. TEST DE INFLACIÓN DE VARIANZA (VIF)")
print("Regla: VIF > 5 indica multicolinealidad severa (las variables se estorban)")
print("="*60)
vif_data = pd.DataFrame()
vif_data["Variable"] = X_con_constante.columns
vif_data["VIF"] = [variance_inflation_factor(X_con_constante.values, i) for i in range(len(X_con_constante.columns))]
display(vif_data.round(2))

# --- PRUEBA 2: EFECTOS MARGINALES ---
print("\n" + "="*60)
print("2. EFECTOS MARGINALES (Traducción al dinero)")
print("Muestra el % exacto que aumenta la probabilidad por cada unidad extra")
print("="*60)
modelo = sm.Logit(Y, X_con_constante).fit(disp=False)
efectos_marginales = modelo.get_margeff(at='overall', method='dydx')
print(efectos_marginales.summary())

# --- PRUEBA 3: PODER PREDICTIVO (MATRIZ DE CONFUSIÓN Y ROC) ---
print("\n" + "="*60)
print("3. DIAGNÓSTICO DE PRECISIÓN PREDICTIVA")
print("="*60)
predicciones_prob = modelo.predict(X_con_constante)
predicciones_binarias = np.where(predicciones_prob > 0.5, 1, 0) # Umbral del 50%

# Matriz de Confusión
tn, fp, fn, tp = confusion_matrix(Y, predicciones_binarias).ravel()
accuracy = accuracy_score(Y, predicciones_binarias)
roc_auc = roc_auc_score(Y, predicciones_prob)

print(f"Predicciones Correctas (Exactitud global): {accuracy:.2%}")
print(f"Poder de Discriminación (ROC-AUC): {roc_auc:.4f} (1.0 es perfecto, 0.5 es azar)")
print(f"\nDesglose de la Matriz de Confusión:")
print(f"- Verdaderos Positivos (Predijo éxito y hay éxito): {tp}")
print(f"- Verdaderos Negativos (Predijo fracaso y no hay nada): {tn}")
print(f"- Falsos Positivos (Predijo éxito pero NO hay competencia): {fp} <-- AQUÍ ESTÁN NUESTROS OCÉANOS AZULES")
print(f"- Falsos Negativos (Predijo fracaso pero SÍ hay competencia): {fn}")

Iniciando Pruebas de Rigor Econométrico...

Nota: Variable 'parque_vehicular' no encontrada, usando modelo base.
1. TEST DE INFLACIÓN DE VARIANZA (VIF)
Regla: VIF > 5 indica multicolinealidad severa (las variables se estorban)


,Variable,VIF
0,const,9.75
1,ingreso_miles,1.11
2,b2b_miles,1.11



2. EFECTOS MARGINALES (Traducción al dinero)
Muestra el % exacto que aumenta la probabilidad por cada unidad extra
        Logit Marginal Effects       
Dep. Variable:      Y_mercado_probado
Method:                          dydx
At:                           overall
                   dy/dx    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
ingreso_miles     0.0015      0.001      1.665      0.096      -0.000       0.003
b2b_miles         0.0703      0.014      5.144      0.000       0.044       0.097

3. DIAGNÓSTICO DE PRECISIÓN PREDICTIVA
Predicciones Correctas (Exactitud global): 89.80%
Poder de Discriminación (ROC-AUC): 0.9181 (1.0 es perfecto, 0.5 es azar)

Desglose de la Matriz de Confusión:
- Verdaderos Positivos (Predijo éxito y hay éxito): 12
- Verdaderos Negativos (Predijo fracaso y no hay nada): 76
- Falsos Positivos (Predijo éxito pero NO hay competencia): 3 <-- AQUÍ ESTÁN NUESTROS OCÉA